# Seminar 2. Custom PyTorch Operators


# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular
- Reusable
- Readable
- Easy to extend

## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [8]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us


### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.


In [9]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.


In [10]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.1278,  0.1570,  0.0072, -0.0190,  0.1655,  0.1562,  0.1097,  0.2430],
        [ 0.0336,  0.1618,  0.1843,  0.0754,  0.2111,  0.1886,  0.2014,  0.2388]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.


In [11]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cuda:0
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.


In [12]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type  | Behavior in `train()` Mode                 |
| ----------- | ------------------------------------------ |
| `Dropout`   | Randomly zeroes some activations           |
| `BatchNorm` | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type  | Behavior in `eval()` Mode                |
| ----------- | ---------------------------------------- |
| `Dropout`   | Passes all activations through unchanged |
| `BatchNorm` | Uses stored running mean/variance        |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.


In [13]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.7777, -0.2645],
        [-0.5735,  0.2931],
        [-1.5036, -0.5298],
        [-0.7993, -0.2359],
        [-0.1667,  0.0885]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.1622,  0.2589],
        [-0.2051,  0.1872],
        [-0.2793,  0.1259],
        [-0.1054,  0.0707],
        [ 0.1222,  0.1669]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.


In [14]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[ 0.4976,  0.1814],
        [ 0.4151, -1.2044],
        [-0.2831,  0.8040],
        [ 0.5035,  0.3422],
        [ 0.6620,  0.1670]])
Output inference_mode method:
 tensor([[ 0.4976,  0.1814],
        [ 0.4151, -1.2044],
        [-0.2831,  0.8040],
        [ 0.5035,  0.3422],
        [ 0.6620,  0.1670]])
Output no_grad context manager:
 tensor([[ 0.4976,  0.1814],
        [ 0.4151, -1.2044],
        [-0.2831,  0.8040],
        [ 0.5035,  0.3422],
        [ 0.6620,  0.1670]])
Output inference_mode context manager:
 tensor([[ 0.4976,  0.1814],
        [ 0.4151, -1.2044],
        [-0.2831,  0.8040],
        [ 0.5035,  0.3422],
        [ 0.6620,  0.1670]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [15]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [16]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.0000, 1.2221],
        [0.0000, 2.3026, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.4755, 0.0000, 0.6111],
        [0.4122, 1.1513, 0.0000, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:


## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.


In [17]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.


In [18]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.


In [19]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Работа на семинаре


### LSTM

![lstm](assets/LSTM.png)


In [34]:
class LSTMCell(nn.Module):
    """
    Одна ячейка LSTM по схеме из семинара.
    
    Входы: x_t, h_{t-1}, c_{t-1}
    Выходы: h_t, c_t
    """
    
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        # [h_{t-1}, x_t] -> размер входа: hidden_size + input_size
        combined_size = input_size + hidden_size
        
        # Forget gate: σ(W_f @ [h, x])
        self.W_f = nn.Linear(combined_size, hidden_size)
        # Input gate: σ(W_i @ [h, x])
        self.W_i = nn.Linear(combined_size, hidden_size)
        # Candidate cell: tanh(W_c @ [h, x])
        self.W_c = nn.Linear(combined_size, hidden_size)
        # Output gate: σ(W_o @ [h, x])
        self.W_o = nn.Linear(combined_size, hidden_size)
    
    def forward(
        self,
        x: torch.Tensor,
        h_prev: torch.Tensor,
        c_prev: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        x: (batch, input_size)
        h_prev, c_prev: (batch, hidden_size)
        """
        combined = torch.cat([h_prev, x], dim=1)  # (batch, input_size + hidden_size)
        
        f_t = torch.sigmoid(self.W_f(combined))   # forget gate
        i_t = torch.sigmoid(self.W_i(combined))   # input gate
        c_tilde = torch.tanh(self.W_c(combined))  # candidate cell state
        o_t = torch.sigmoid(self.W_o(combined))   # output gate
        
        c_t = f_t * c_prev + i_t * c_tilde        # new cell state
        h_t = o_t * torch.tanh(c_t)               # new hidden state
        
        return h_t, c_t


class LSTMLayer(nn.Module):
    """
    Один слой LSTM: последовательно применяет LSTMCell по всем шагам времени.
    """
    
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)
        self.hidden_size = hidden_size
    
    def forward(
        self,
        x: torch.Tensor,
        h_0: torch.Tensor | None = None,
        c_0: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        x: (batch, seq_len, input_size)
        h_0, c_0: (batch, hidden_size) — начальные состояния (опционально)
        Возвращает: output (batch, seq_len, hidden_size), h_n, c_n
        """
        batch, seq_len, _ = x.shape
        device = x.device
        
        if h_0 is None:
            h = torch.zeros(batch, self.hidden_size, device=device, dtype=x.dtype)
        else:
            h = h_0
        if c_0 is None:
            c = torch.zeros(batch, self.hidden_size, device=device, dtype=x.dtype)
        else:
            c = c_0
        
        outputs = []
        for t in range(seq_len):
            h, c = self.cell(x[:, t, :], h, c)
            outputs.append(h)
        
        return torch.stack(outputs, dim=1), h, c


# Проверка
if __name__ == "__main__":
    batch, seq_len, input_size, hidden_size = 2, 5, 10, 8
    x = torch.randn(batch, seq_len, input_size)
    
    lstm = LSTMLayer(input_size, hidden_size)
    output, h_n, c_n = lstm(x)
    
    print("Input shape:", x.shape)
    print("Output shape:", output.shape)
    print("h_n shape:", h_n.shape)
    print("c_n shape:", c_n.shape)

Input shape: torch.Size([2, 5, 10])
Output shape: torch.Size([2, 5, 8])
h_n shape: torch.Size([2, 8])
c_n shape: torch.Size([2, 8])


### Inception

![inception](assets/inception.png)


In [ ]:
class InceptionModule(nn.Module):
    """
    Inception module с dimension reduction (схема b из статьи).
    
    4 параллельных ветки:
    - 1×1 conv
    - 1×1 (bottleneck) → 3×3 conv
    - 1×1 (bottleneck) → 5×5 conv
    - 3×3 max pool → 1×1 conv (reduction)
    
    Выходы всех веток конкатенируются по каналам.
    """
    
    def __init__(
        self,
        in_channels: int,
        out_1x1: int,
        out_3x3_reduce: int,
        out_3x3: int,
        out_5x5_reduce: int,
        out_5x5: int,
        out_pool: int,
    ) -> None:
        super().__init__()
        
        # Ветка 1: только 1×1 conv
        self.branch1 = nn.Conv2d(in_channels, out_1x1, kernel_size=1)
        
        # Ветка 2: 1×1 (reduction) → 3×3 conv
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_3x3_reduce, kernel_size=1),
            nn.Conv2d(out_3x3_reduce, out_3x3, kernel_size=3, padding=1),  # padding для сохранения размера
        )
        
        # Ветка 3: 1×1 (reduction) → 5×5 conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_5x5_reduce, kernel_size=1),
            nn.Conv2d(out_5x5_reduce, out_5x5, kernel_size=5, padding=2),  # padding для сохранения размера
        )
        
        # Ветка 4: 3×3 max pool → 1×1 conv (reduction)
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),  # stride=1, padding для сохранения размера
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, in_channels, H, W)
        """
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


# Пример использования (параметры из GoogLeNet)
def create_inception_module_example(in_channels: int = 256) -> InceptionModule:
    """
    Пример конфигурации, аналогичной Inception из GoogLeNet.
    """
    return InceptionModule(
        in_channels=in_channels,
        out_1x1=64,
        out_3x3_reduce=96,
        out_3x3=128,
        out_5x5_reduce=16,
        out_5x5=32,
        out_pool=32,
    )


# Проверка
if __name__ == "__main__":
    module = InceptionModule(
        in_channels=256,
        out_1x1=64,
        out_3x3_reduce=96,
        out_3x3=128,
        out_5x5_reduce=16,
        out_5x5=32,
        out_pool=32,
    )
    
    x = torch.randn(2, 256, 28, 28)
    out = module(x)
    
    print("Input shape:", x.shape)
    print("Output shape:", out.shape)
    # Output: (2, 64+128+32+32, 28, 28) = (2, 256, 28, 28)

Input shape: torch.Size([2, 256, 28, 28])
Output shape: torch.Size([2, 256, 28, 28])


### SE

![se](assets/SqueezeAndExcite.png)


In [ ]:
class SqueezeExcitation(nn.Module):
    """
    Squeeze-and-Excitation block.
    
    1. Squeeze (F_sq): global average pooling → channel descriptors 1×1×C
    2. Excitation (F_ex): FC → ReLU → FC → Sigmoid → channel weights
    3. Scale (F_scale): element-wise умножение U на веса каналов
    """
    
    def __init__(self, channels: int, reduction: int = 16) -> None:
        super().__init__()
        self.channels = channels
        reduced = max(channels // reduction, 8)  # не меньше 8 каналов
        
        self.squeeze = nn.AdaptiveAvgPool2d(1)  # (B,C,H,W) → (B,C,1,1)
        self.excitation = nn.Sequential(
            nn.Conv2d(channels, reduced, kernel_size=1),  # FC as 1×1 conv
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced, channels, kernel_size=1),
            nn.Sigmoid(),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, C, H, W)
        """
        # Squeeze
        squeezed = self.squeeze(x)  # (B, C, 1, 1)
        # Excitation
        weights = self.excitation(squeezed)  # (B, C, 1, 1)
        # Scale
        return x * weights


class SEBlock(nn.Module):
    """
    Полный SE-блок: Conv → SE.    
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        reduction: int = 16,
        kernel_size: int = 3,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size, padding=kernel_size // 2
        )
        self.se = SqueezeExcitation(out_channels, reduction)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = self.conv(x)
        return self.se(u)


# Проверка
if __name__ == "__main__":
    # Вариант 1: только SE (когда U уже есть)
    se = SqueezeExcitation(channels=256, reduction=16)
    x = torch.randn(4, 256, 14, 14)
    out = se(x)
    print("SE only - Input:", x.shape, "Output:", out.shape)
    
    # Вариант 2: Conv + SE
    block = SEBlock(in_channels=64, out_channels=256, reduction=16)
    x = torch.randn(4, 64, 28, 28)
    out = block(x)
    print("SEBlock - Input:", x.shape, "Output:", out.shape)

SE only - Input: torch.Size([4, 256, 14, 14]) Output: torch.Size([4, 256, 14, 14])
SEBlock - Input: torch.Size([4, 64, 28, 28]) Output: torch.Size([4, 256, 28, 28])


### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [ ]:
class SelectiveKernelConv(nn.Module):
    """
    Selective Kernel Convolution.
    
    1. Split: 3×3 и 5×5 свёртки → Ũ, Û
    2. Fuse: U = Ũ + Û, GAP → s, FC → z
    3. Generate: z → (a, b), softmax по веткам
    4. Select: V = a ⊗ Ũ + b ⊗ Û
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        reduction: int = 16,
    ) -> None:
        super().__init__()
        mid_channels = max(out_channels // reduction, 32)
        
        # Split
        self.branch_3x3 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.branch_5x5 = nn.Conv2d(in_channels, out_channels, 5, padding=2)
        
        # Fuse + Generate
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(out_channels, mid_channels, 1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels * 2, 1),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Split
        u_tilde = self.branch_3x3(x)
        u_hat = self.branch_5x5(x)
        
        # Fuse
        u = u_tilde + u_hat
        s = self.pool(u)
        z = self.fc(s)  # (B, C*2, 1, 1)
        
        # Generate: softmax по двум веткам
        batch, c2, _, _ = z.shape
        z = z.view(batch, 2, c2 // 2)
        attn = z.softmax(dim=1)
        # (B, C, 1, 1) для channel-wise масштабирования (B, C, H, W)
        a = attn[:, 0].unsqueeze(-1).unsqueeze(-1)
        b = attn[:, 1].unsqueeze(-1).unsqueeze(-1)
        
        # Select
        return a * u_tilde + b * u_hat


# Проверка
if __name__ == "__main__":
    sk = SelectiveKernelConv(64, 128, reduction=16)
    x = torch.randn(4, 64, 28, 28)
    out = sk(x)
    print("Input:", x.shape, "Output:", out.shape)

Input: torch.Size([4, 64, 28, 28]) Output: torch.Size([4, 128, 28, 28])


### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [26]:
class PatchMerger(nn.Module):
    """
    PatchMerger: N токенов (N, D) → M токенов (M, D).
    
    1. LayerNorm над входом
    2. Learned Weights (M, D): M «запросов»
    3. scores = x_norm @ W^T → (N, M)
    4. transpose + softmax → attention (M, N)
    5. output = attn @ x_norm → (M, D)
    """
    
    def __init__(self, num_input_patches: int, num_output_patches: int, hidden_dim: int) -> None:
        super().__init__()
        self.n_in = num_input_patches
        self.n_out = num_output_patches
        self.d = hidden_dim
        
        self.norm = nn.LayerNorm(hidden_dim)
        self.weights = nn.Parameter(torch.randn(num_output_patches, hidden_dim) * 0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, N, D)
        return: (batch, M, D)
        """
        x_norm = self.norm(x)
        # scores: (B, N, M) — сходство каждого входа с каждым «запросом»
        scores = torch.matmul(x_norm, self.weights.T)
        # attn: (B, M, N), softmax по N — веса агрегации для каждого выхода
        attn = scores.permute(0, 2, 1).softmax(dim=-1)
        out = torch.matmul(attn, x_norm)
        return out


# Проверка
if __name__ == "__main__":
    B, N, M, D = 4, 64, 16, 128
    x = torch.randn(B, N, D)
    merger = PatchMerger(num_input_patches=N, num_output_patches=M, hidden_dim=D)
    out = merger(x)
    print("Input:", x.shape, "→ Output:", out.shape)

Input: torch.Size([4, 64, 128]) → Output: torch.Size([4, 16, 128])


## Homework

2 задания:

1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).


In [29]:
# LSTM

class LSTMCell(nn.Module):
    """
    Одна ячейка LSTM по схеме из семинара.
    
    Входы: x_t, h_{t-1}, c_{t-1}
    Выходы: h_t, c_t
    """
    
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        # [h_{t-1}, x_t] -> размер входа: hidden_size + input_size
        combined_size = input_size + hidden_size
        
        # Forget gate: σ(W_f @ [h, x])
        self.W_f = nn.Linear(combined_size, hidden_size)
        # Input gate: σ(W_i @ [h, x])
        self.W_i = nn.Linear(combined_size, hidden_size)
        # Candidate cell: tanh(W_c @ [h, x])
        self.W_c = nn.Linear(combined_size, hidden_size)
        # Output gate: σ(W_o @ [h, x])
        self.W_o = nn.Linear(combined_size, hidden_size)
    
    def forward(
        self,
        x: torch.Tensor,
        h_prev: torch.Tensor,
        c_prev: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        x: (batch, input_size)
        h_prev, c_prev: (batch, hidden_size)
        """
        combined = torch.cat([h_prev, x], dim=1)  # (batch, input_size + hidden_size)
        
        f_t = torch.sigmoid(self.W_f(combined))   # forget gate
        i_t = torch.sigmoid(self.W_i(combined))   # input gate
        c_tilde = torch.tanh(self.W_c(combined))  # candidate cell state
        o_t = torch.sigmoid(self.W_o(combined))   # output gate
        
        c_t = f_t * c_prev + i_t * c_tilde        # new cell state
        h_t = o_t * torch.tanh(c_t)               # new hidden state
        
        return h_t, c_t


class LSTMLayer(nn.Module):
    """
    Один слой LSTM: последовательно применяет LSTMCell по всем шагам времени.
    """
    
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)
        self.hidden_size = hidden_size
    
    def forward(
        self,
        x: torch.Tensor,
        h_0: torch.Tensor | None = None,
        c_0: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        x: (batch, seq_len, input_size)
        h_0, c_0: (batch, hidden_size) — начальные состояния (опционально)
        Возвращает: output (batch, seq_len, hidden_size), h_n, c_n
        """
        batch, seq_len, _ = x.shape
        device = x.device
        
        if h_0 is None:
            h = torch.zeros(batch, self.hidden_size, device=device, dtype=x.dtype)
        else:
            h = h_0
        if c_0 is None:
            c = torch.zeros(batch, self.hidden_size, device=device, dtype=x.dtype)
        else:
            c = c_0
        
        outputs = []
        for t in range(seq_len):
            h, c = self.cell(x[:, t, :], h, c)
            outputs.append(h)
        
        return torch.stack(outputs, dim=1), h, c


# Проверка
if __name__ == "__main__":
    batch, seq_len, input_size, hidden_size = 2, 5, 10, 8
    x = torch.randn(batch, seq_len, input_size)
    
    lstm = LSTMLayer(input_size, hidden_size)
    output, h_n, c_n = lstm(x)
    
    print("Input shape:", x.shape)
    print("Output shape:", output.shape)
    print("h_n shape:", h_n.shape)
    print("c_n shape:", c_n.shape)

Input shape: torch.Size([2, 5, 10])
Output shape: torch.Size([2, 5, 8])
h_n shape: torch.Size([2, 8])
c_n shape: torch.Size([2, 8])


## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385


In [31]:
class ResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
    ) -> None:
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )        
        
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if in_channels != out_channels or stride != 1
            else nn.Identity()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        out = self.main(x)
        return self.relu(out + identity)

# Проверка
if __name__ == "__main__":
    block = ResidualBlock(64, 128, stride=2)
    x = torch.randn(4, 64, 32, 32)
    out = block(x)
    print("Input:", x.shape, "→ Output:", out.shape)

Input: torch.Size([4, 64, 32, 32]) → Output: torch.Size([4, 128, 16, 16])


## Depthwise Separable Convolution (0.1 балл)

![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357


In [32]:
class SeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int | None = None,
        bias: bool = False,
    ) -> None:
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=bias,
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        return self.pointwise(x)


# Проверка
if __name__ == "__main__":
    conv = SeparableConv2d(64, 128, kernel_size=3, stride=1)
    x = torch.randn(4, 64, 32, 32)
    out = conv(x)
    print("Input:", x.shape, "→ Output:", out.shape)

Input: torch.Size([4, 64, 32, 32]) → Output: torch.Size([4, 128, 32, 32])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$

https://arxiv.org/abs/1409.0473

https://arxiv.org/abs/1508.04025


In [35]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d: int) -> None:
        super().__init__()
        self.W_align = nn.Linear(d, d)
        self.W_value = nn.Linear(d, d)
        self.W_query = nn.Linear(d, d)
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
    ) -> torch.Tensor:
        """
        query: (B, d)
        key: (B, L, d)
        return: (B, d)
        """
        B, L, _ = key.shape
        query_proj = self.W_align(query)  # (B, d)
        score = torch.bmm(key, query_proj.unsqueeze(-1)).squeeze(-1)  # (B, L)
        att = torch.softmax(score, dim=1)    # (B, L)
        context = (att.unsqueeze(-1) * key).sum(dim=1)  # (B, d)
        out = torch.tanh(self.W_value(context) + self.W_query(query))
        return out


# Проверка
if __name__ == "__main__":
    B, L, d = 4, 10, 32
    attn = VanillaAttention(d)
    query = torch.randn(B, d)
    key = torch.randn(B, L, d)
    out = attn(query, key)
    print("query:", query.shape, "key:", key.shape, "→ out:", out.shape)

query: torch.Size([4, 32]) key: torch.Size([4, 10, 32]) → out: torch.Size([4, 32])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$

https://arxiv.org/abs/1706.03762


In [36]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        query: (B, L_q, d_k)
        key:   (B, L_k, d_k)
        value: (B, L_k, d_v)
        mask:  (B, L_q, L_k) или (L_q, L_k), опционально
        
        return: (output, attention_weights)
        output: (B, L_q, d_v)
        attention_weights: (B, L_q, L_k)
        """
        d_k = query.size(-1)
        scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)  # (B, L_q, L_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        output = torch.matmul(attn, value)  # (B, L_q, d_v)
        return output, attn


# Проверка
if __name__ == "__main__":
    B, L_q, L_k, d_k, d_v = 4, 10, 12, 64, 128
    attn = ScaledDotProductAttention(dropout=0.0)
    q = torch.randn(B, L_q, d_k)
    k = torch.randn(B, L_k, d_k)
    v = torch.randn(B, L_k, d_v)
    out, weights = attn(q, k, v)
    print("output:", out.shape, "| attention:", weights.shape)

output: torch.Size([4, 10, 128]) | attention: torch.Size([4, 10, 12])


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [37]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        query: (B, L_q, d_model)
        key:   (B, L_k, d_model)
        value: (B, L_k, d_model)
        return: (output, attention_weights)
        """
        B, L_q, _ = query.shape
        _, L_k, _ = key.shape
        
        Q = self.W_q(query)   # (B, L_q, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # Reshape для h голов: (B, L, d_model) → (B, h, L, d_k)
        Q = Q.view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (B, h, L_q, L_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float("-inf"))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, V)  # (B, h, L_q, d_k)
        out = out.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        
        return self.W_o(out), attn


# Проверка
if __name__ == "__main__":
    B, L_q, L_k, d_model, h = 4, 10, 12, 64, 8
    mha = MultiHeadAttention(d_model=d_model, num_heads=h)
    q = torch.randn(B, L_q, d_model)
    k = torch.randn(B, L_k, d_model)
    v = torch.randn(B, L_k, d_model)
    out, weights = mha(q, k, v)
    print("output:", out.shape, "| attention:", weights.shape)

output: torch.Size([4, 10, 64]) | attention: torch.Size([4, 8, 10, 12])


## Transformer Encoder Layer (0.1 балл)

![Transformer Encoder Layer](assets/TransformerEncoder.png)

https://arxiv.org/abs/1706.03762


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
        padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        x: (B, L, d_model)
        mask: (L, L) — attention mask (causal/padding)
        padding_mask: (B, L) — True = mask out
        """
        # Sub-layer 1: Self-Attention + residual
        x_norm = self.norm1(x)
        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm, attn_mask=mask, key_padding_mask=padding_mask)
        x = x + attn_out
        
        # Sub-layer 2: MLP + residual
        x = x + self.ff(self.norm2(x))
        
        return x

# Проверка
if __name__ == "__main__":
    B, L, d_model, h, d_ff = 4, 20, 64, 8, 256
    layer = TransformerEncoderLayer(d_model=d_model, num_heads=h, d_ff=d_ff)
    x = torch.randn(B, L, d_model)
    out = layer(x)
    print("Input:", x.shape, "→ Output:", out.shape)

Input: torch.Size([4, 20, 64]) → Output: torch.Size([4, 20, 64])


## MLP Mixer (0.1 балл)

![MLPMixer](assets/MLPMixer.png)

https://arxiv.org/abs/2105.01601


In [39]:


class MLPMixerBlock(nn.Module):
    def __init__(
        self,
        num_patches: int,
        channels: int,
        token_dim: int,
        channel_dim: int,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_dim),
            nn.GELU(),
            nn.Linear(token_dim, num_patches),
        )
        self.channel_mlp = nn.Sequential(
            nn.Linear(channels, channel_dim),
            nn.GELU(),
            nn.Linear(channel_dim, channels),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, P, C) — patches, channels
        """
        # Token-mixing: смешивание по патчам (транспорт, MLP по P, транспорт обратно)
        x = x + self.token_mlp(self.norm1(x).transpose(-2, -1)).transpose(-2, -1)
        # Channel-mixing: смешивание по каналам
        x = x + self.channel_mlp(self.norm2(x))
        return x


# Проверка
if __name__ == "__main__":
    B, P, C = 4, 64, 128
    token_dim, channel_dim = 256, 512
    block = MLPMixerBlock(num_patches=P, channels=C, token_dim=token_dim, channel_dim=channel_dim)
    x = torch.randn(B, P, C)
    out = block(x)
    print("Input:", x.shape, "→ Output:", out.shape)


Input: torch.Size([4, 64, 128]) → Output: torch.Size([4, 64, 128])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)

https://arxiv.org/abs/2201.09792


In [40]:
class ConvMixer(nn.Module):

    def __init__(
        self,
        channels: int,
        kernel_size: int = 9,
        expansion: int = 4,
    ) -> None:
        super().__init__()
        hidden = channels * expansion
        # Depthwise: Conv → ReLU → BN (+ residual)
        self.depthwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=kernel_size // 2, groups=channels),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(channels),
        )
        # Pointwise: Conv 1×1 → ReLU → BN
        self.pointwise = nn.Sequential(
            nn.Conv2d(channels, channels, 1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.depthwise(x)
        x = self.pointwise(x)
        return x


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Все три выполняют смешивание информации между токенами/патчами:

- Attention — через взвешенное суммирование на основе сходства
- MLPMixer — через линейные слои (token-mixing и channel-mixing)
- ConvMixer — через depthwise (пространственное смешивание) и pointwise (смешивание каналов)
  Важен не конкретный механизм (attention vs MLP vs conv), а возможность обмена информацией между позициями и каналами.

**Формула, связывающая все три:**

Общая форма:

$$\text{out} = \text{Aggregate}(\text{mixing}(\text{in}), \text{in})$$

Где:

Multihead Attention:

- $\text{mixing}(X) = \text{softmax}(QK^T/\sqrt{d})V$,
- $\text{Aggregate} = \text{add}$ (residual)

MLPMixer:

- $\text{mixing}(X) = \text{MLP}C(\text{MLP}_P(X^T)^T)$,
- $\text{Aggregate} = \text{add}$

ConvMixer:

- $\text{mixing}(X) = \text{Pointwise}(\text{Depthwise}(X))$,
- $\text{Aggregate} = \text{add}$ (внутри блока)

Общее:

$$\text{Output} = X + f(X), \quad f = \text{mixing}.$$

**Преимущества и недостатки**

**Multihead Attention** даёт сильную индуктивную bias на длинные зависимости и позволяет анализировать веса внимания, но имеет квадратичную по длине последовательности сложность по памяти и вычислениям O(L²).

**MLPMixer** работает за O(L) по длине, проще в реализации и не использует позиционное кодирование. Однако у него фиксированный receptive field, и на очень длинных последовательностях он может проигрывать attention.

**ConvMixer** имеет сложность O(1) по длине последовательности, использует локальную связность и хорошо масштабируется. Минусы: ограниченный локальным receptive field, который расширяется только при увеличении размера kernel.
